# Hypothesis Testing & p-values

Companion notebook for the [Hypothesis Testing lesson](https://ml-viz.vercel.app/courses/probability-statistics/07-hypothesis-testing).

We implement a **z-test from scratch**, visualize the p-value and rejection region on the null
distribution, confirm the false-positive rate equals α by simulation, and measure **statistical
power** as a function of effect size. Pure NumPy + Matplotlib (a hand-rolled normal CDF, so no SciPy
needed).

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d27',
    'axes.edgecolor': '#444', 'axes.labelcolor': '#ccc',
    'xtick.color': '#888', 'ytick.color': '#888',
    'text.color': '#eee', 'grid.color': '#333', 'lines.linewidth': 2,
})
rng = np.random.default_rng(0)

def normal_cdf(z):
    """Standard-normal CDF via the error function."""
    return 0.5 * (1 + erf(z / sqrt(2)))

## 1 — A z-test from scratch

Given a null mean μ₀ and known σ, the z-statistic is `z = (x̄ − μ₀)/(σ/√n)`, standard-normal under
H₀. The two-sided p-value is the mass beyond `|z|` in both tails.

In [ ]:
def z_test(xbar, mu0, sigma, n, two_tailed=True):
    se = sigma / np.sqrt(n)
    z = (xbar - mu0) / se
    if two_tailed:
        p = 2 * (1 - normal_cdf(abs(z)))
    else:
        p = 1 - normal_cdf(z)
    return z, p

# observed mean 105 vs null 100, sigma 20, n 64
z, p = z_test(105, 100, 20, 64)
print(f"z = {z:.3f},  two-tailed p = {p:.4f}")
print("reject H0 at alpha=0.05" if p < 0.05 else "fail to reject H0")

## 2 — Visualizing the p-value and rejection region

Blue = the p-value (mass beyond the observed |z|); rose = the rejection region (the α-tails). You
reject H₀ exactly when |z| exceeds the critical value z* — i.e. when the blue extends past the rose.

In [ ]:
z_obs, alpha = 2.0, 0.05
z_crit = 1.96   # two-tailed critical value for alpha=0.05
x = np.linspace(-4, 4, 500)
pdf = np.exp(-0.5 * x**2) / np.sqrt(2 * np.pi)

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(x, pdf, color='#e2e8f0')
ax.fill_between(x, pdf, where=np.abs(x) >= z_crit, color='#fb7185', alpha=0.35, label=f'rejection region (alpha={alpha})')
ax.fill_between(x, pdf, where=np.abs(x) >= z_obs, color='#6366f1', alpha=0.55, label=f'p-value (|z|>={z_obs})')
ax.axvline(z_obs, color='#eab308', label=f'observed z = {z_obs}')
ax.set_xlabel('test statistic z'); ax.set_ylabel('density')
ax.set_title('p-value (blue) vs rejection region (rose) under H0')
ax.legend(facecolor='#1a1d27', edgecolor='#444', fontsize=8)
plt.tight_layout(); plt.show()

## 3 — The false-positive rate equals α

We simulate experiments where H₀ is **true** (data drawn at the null mean) and count how often we
wrongly reject at α = 0.05. The Type I error rate should land right at 5%.

In [ ]:
mu0, sigma, n, trials, alpha = 100, 20, 64, 20000, 0.05
false_pos = 0
for _ in range(trials):
    sample = rng.normal(mu0, sigma, size=n)      # H0 is TRUE
    _, p = z_test(sample.mean(), mu0, sigma, n)
    if p < alpha:
        false_pos += 1
print(f"Type I error rate: {100*false_pos/trials:.2f}%  (equals alpha = {alpha*100:.0f}%)")

## 4 — Statistical power vs. effect size

Power = P(reject H₀ | H₀ false) — the chance of catching a real effect. We sweep the true effect
(how far the real mean sits from μ₀) and estimate how often the test correctly rejects. Bigger
effects (and bigger n) give more power.

In [ ]:
def power(effect, mu0=100, sigma=20, n=64, alpha=0.05, trials=3000):
    rejects = 0
    for _ in range(trials):
        sample = rng.normal(mu0 + effect, sigma, size=n)   # H0 is FALSE
        _, p = z_test(sample.mean(), mu0, sigma, n)
        if p < alpha:
            rejects += 1
    return rejects / trials

effects = np.array([0, 2, 4, 6, 8, 10, 12])
powers = [power(e) for e in effects]

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(effects, powers, 'o-', color='#2dd4bf')
ax.axhline(0.8, ls='--', color='#555', label='conventional 80% power target')
ax.axhline(0.05, ls=':', color='#fb7185', label='alpha (effect=0 -> Type I rate)')
ax.set_xlabel('true effect size (mean shift)'); ax.set_ylabel('power = P(reject | H1 true)')
ax.set_title('Bigger effects are easier to detect (higher power)')
ax.legend(facecolor='#1a1d27', edgecolor='#444', fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"effect=0  -> power {powers[0]:.2f}  (this is just the false-positive rate alpha)")
print(f"effect=8  -> power {power(8):.2f}")

## ✏️ Your turn

**Exercise.** Implement `decision(p, alpha)` returning `"reject H0"` when significant else
`"fail to reject H0"`, and `two_sample_z(xbarA, xbarB, se_a, se_b)` returning the two-sample
z-statistic `(x̄_B − x̄_A)/√(SE_A² + SE_B²)` used to compare two groups (e.g. an A/B test).

In [ ]:
def decision(p, alpha=0.05):
    # TODO(you): return 'reject H0' or 'fail to reject H0'
    return ...

def two_sample_z(xbarA, xbarB, se_a, se_b):
    # TODO(you): standardized difference between two group means
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert decision(0.03) == 'reject H0'
assert decision(0.20) == 'fail to reject H0'
# two groups: B higher by 5, each SE = 3 -> z = 5/sqrt(18) ~ 1.179
z = two_sample_z(50, 55, 3, 3)
assert abs(z - 5/np.sqrt(18)) < 1e-9
# single-test Type I rate check from section 3
assert 0.03 < false_pos/trials < 0.07
print(f"\u2713 decision rule and two-sample z (= {z:.3f}) are correct")

<details>
<summary>Solution</summary>

```python
def decision(p, alpha=0.05):
    return 'reject H0' if p < alpha else 'fail to reject H0'

def two_sample_z(xbarA, xbarB, se_a, se_b):
    return (xbarB - xbarA) / np.sqrt(se_a**2 + se_b**2)
```

The two-sample z-test is the backbone of A/B testing: standardize the observed difference between
the groups by the combined standard error, then read off a p-value. A large |z| means the gap is far
more than sampling noise would explain.

</details>